# OP26 — Phase 3: Demand Prediction Agent
Forecasts, **one hour ahead** per zone: utilization, expected load (energy), and **congestion probability** P(util ≥ 0.80). Logic in `demand.py`.

**Leakage-safe:** the target is the value at hour *t*; every feature is strictly pre-*t* (lags / rolling) or deterministic for *t* (calendar / static zone attributes). Contemporaneous outcomes (occupancy, energy, revenue, price at *t*) are never used. Validation is a strict **time-based split** — the final 4 days are the untouched test set. Gradient boosting = sklearn HistGradientBoosting (same GBM family as LightGBM, available offline).

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from IPython.display import Image, display
import config as C, demand as D
train, test = D.load_split()
print('train rows', len(train), '| test rows', len(test),
      f'| test = hour_index >= {D.TEST_START} (final {720-D.TEST_START}h / 4 days)')

## Train all three heads, score, and write outputs
`D.run()` fits the utilization regressor, the energy (load) regressor, and the congestion classifier; computes baselines; and writes predictions, metrics, per-zone metrics, feature importance, models, and figures.

In [ ]:
metrics, preds, fi, by_zone = D.run()
# headline metrics
piv = (metrics[metrics.target=='utilization']
       .pivot(index='model', columns='metric', values='value')
       .sort_values('RMSE'))
display(piv)
print('Energy load  :', metrics[(metrics.target=='energy_kwh')&(metrics.model=='GBM')]
      .set_index('metric').value.round(3).to_dict())
print('Congestion   :', metrics[(metrics.target=='congestion')&(metrics.model=='GBM')]
      .set_index('metric').value.round(3).to_dict())

**Result:** the GBM beats every baseline on utilization (incl. a strong 1-hour persistence), explains ~97% of load variance, and detects the rare >80% congestion events with high AUC. Recent lags + the weekly lag + zone identity dominate — consistent with the Phase-2 seasonality.

In [ ]:
display(Image(C.FIG_DIR/'fig09_demand_pred.png'))
display(Image(C.FIG_DIR/'fig11_congestion_roc.png'))

## Per-zone performance & feature importance

In [ ]:
print('per-zone GBM utilization: median R2 = %.3f | mean RMSE = %.4f | zones R2<0: %d'
      % (by_zone.R2.median(), by_zone.RMSE.mean(), (by_zone.R2<0).sum()))
display(by_zone.sort_values('util_mean', ascending=False).head(8))
display(fi.head(10))
display(Image(C.FIG_DIR/'fig10_feature_importance.png'))

---
**Phase 3 complete.** `demand_predictions.csv` (forecast utilization + congestion probability per zone-hour) is the direct input to **Phase 4 — the Tariff Pricing Agent**.